First we do some quality checks with the data compared to the original intervals.

In [1]:
import numpy as np
from Historia.shared.design_utils import read_labels

data_path_new = f"/media/croderog/SeagateExpansionDrive/rodero_healthy/h11/scenarios/5/MCMC_4timesUpper/data"
X_file_new_name = f"{data_path_new}/MCMC_samples_scenario_5_50_4timesUpper.dat"

data_path_old = f"/media/croderog/SeagateExpansionDrive/rodero_healthy/h11/scenarios/5/data"
X_file_old_name = f"{data_path_old}/X.txt"

X_new = np.loadtxt(X_file_new_name,dtype=float)
X_old = np.loadtxt(X_file_old_name,dtype=float)

xlabels = read_labels(f"{data_path_old}/xlabels.txt")

min_X_new = np.round(np.min(X_new,axis=0),6)
min_X_old = np.round(np.min(X_old,axis=0),6)

max_X_new = np.round(np.max(X_new,axis=0),6)
max_X_old = np.round(np.max(X_old,axis=0),6)

perc_increase_min = np.round(100*(min_X_new-min_X_old)/min_X_old,2)
perc_increase_max = np.round(100*(max_X_new-max_X_old)/max_X_old,2)

for i, label in enumerate(xlabels):
	print(f"The interval for {label} is [{min_X_new[i]}, {max_X_new[i]}]. Compared to the original interval of [{min_X_old[i]}, {max_X_old[i]}] ([{perc_increase_min[i]}%, {perc_increase_max[i]}%])")

The interval for CV_ventricles is [0.604539, 0.655065]. Compared to the original interval of [0.380073, 0.799916] ([59.06%, -18.11%])
The interval for CV_atria is [0.580311, 0.703971]. Compared to the original interval of [0.300655, 1.02952] ([93.02%, -31.62%])
The interval for Rsys is [2.714021, 4.712599]. Compared to the original interval of [1.00061, 3.99788] ([171.24%, 17.88%])
The interval for Rpulm is [0.507969, 1.20067]. Compared to the original interval of [1.00002, 3.99789] ([-49.2%, -69.97%])
The interval for a_ventricles is [0.501491, 1.630929]. Compared to the original interval of [1.00155, 4.99757] ([-49.93%, -67.37%])
The interval for a_atria is [0.700718, 6.877074]. Compared to the original interval of [1.00162, 4.99744] ([-30.04%, 37.61%])
The interval for EDP_lv is [17.89801, 26.92618]. Compared to the original interval of [1.00254, 7.4972] ([1685.27%, 259.15%])
The interval for EDP_rv is [0.620465, 4.397982]. Compared to the original interval of [1.00051, 7.49911] ([-

First we need to split the X file into the different fields.

In [2]:
from SIMULATION_library import simulator_utils

fields = ["EP", "circadapt", "mechanics"]

simulator_utils.split_X(X_file = X_file_new_name,
						idx_list = [[0,1],[2,3],[4,5,6,7]],
						X_output_file_list = [f"{data_path_new}/X_{fields[0]}.txt",
											  f"{data_path_new}/X_{fields[1]}.txt",
											  f"{data_path_new}/X_{fields[2]}.txt"])

Now we copy the adequate xlabels and create a new `X.txt`:

In [3]:
import os

X_array = []
xlabels_array = []

for field in fields:
	os.system(f"cp {data_path_old}/xlabels_{field}.txt {data_path_new}/.")
    
	X_ = np.loadtxt(f"{data_path_new}/X_{field}.txt", dtype=float)

    # Check if X_ has only one column, reshape to 2D array
	if X_.ndim == 1:
		X_ = X_.reshape(-1, 1)

	X_array.append(X_)

	xlabels_ = read_labels(f"{data_path_new}/xlabels_{field}.txt")
	xlabels_array.append(xlabels_)

# X = np.concatenate(X_array, axis=1)
    
X = np.hstack(X_array)
xlabels = np.concatenate(xlabels_array, axis=0)

np.savetxt(f"{data_path_new}/X.txt",X,fmt="%g")
np.savetxt(f"{data_path_new}/xlabels.txt",xlabels,fmt="%s")

We create the needed json files:

In [5]:
from SIMULATION_library import simulator_utils

simulator_utils.X_to_json(labels_fields = fields,
                          datafolder    = data_path_new,
                          outputfolder  = f"{data_path_new}/../json_files",
                          default_json  = f"{data_path_new}/../json_files/default.json")

generating json file...
EP
(50, 2)
circadapt
(50, 2)
mechanics
(50, 4)


mkdir: cannot create directory ‘/media/croderog/SeagateExpansionDrive/rodero_healthy/h11/scenarios/5/MCMC_4timesUpper/data/../json_files’: File exists


Now you can run notebooks 1 and 2.